### **Imports**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.datasets import VOCDetection

import cv2
import albumentations as A

import json
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm_notebook as tqdm

from __future__ import division
from math import sqrt as sqrt
from itertools import product as product
from copy import deepcopy

In [ ]:
device=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
# device=torch.device('cpu')
device

May help with `... torch._C._cuda_getDeviceCount() > 0` error

`sudo rmmod nvidia_uvm`  
`sudo modprobe nvidia_uvm`

### **Building SSD 300**

#### **VGG base part**

'M' - max pooling layer

'C' - max pooling layer with **ceil mode** (output size is specified in different way)

In [ ]:
base = {
    '300': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'C', 512, 512, 512, 'M',
            512, 512, 512],
}

In [ ]:
def vgg(cfg, i, batch_norm=False):
    layers = []
    in_channels = i
    for v in cfg:
        if v == 'M':
            layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
        elif v == 'C':
            layers += [nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True)]
        else:
            conv2d = nn.Conv2d(in_channels, v, kernel_size=3, padding=1)
            if batch_norm:
                layers += [conv2d, nn.BatchNorm2d(v), nn.ReLU(inplace=True)]
            else:
                layers += [conv2d, nn.ReLU(inplace=True)]
            in_channels = v
    pool5 = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
    conv6 = nn.Conv2d(512, 1024, kernel_size=3, padding=6, dilation=6)
    conv7 = nn.Conv2d(1024, 1024, kernel_size=1)
    layers += [pool5, conv6,
               nn.ReLU(inplace=True), conv7, nn.ReLU(inplace=True)]
    return layers

#### **Additional feature extractors part**

'S' means convolution layer with stride

In [ ]:
extras = {
    '300': [256, 'S', 512, 128, 'S', 256, 128, 256, 128, 256],
}

In [ ]:
def add_extras(cfg, i, batch_norm=False):
    # Extra layers added to VGG for feature scaling
    layers = []
    in_channels = i
    flag = False
    for k, v in enumerate(cfg):
        if in_channels != 'S':
            if v == 'S':
                layers += [nn.Conv2d(in_channels, cfg[k + 1],
                           kernel_size=(1, 3)[flag], stride=2, padding=1)]
            else:
                layers += [nn.Conv2d(in_channels, v, kernel_size=(1, 3)[flag])]
            flag = not flag
        in_channels = v
    return layers

#### **Convs which outputs are predictions**

In [ ]:
mbox = {
    '300': [4, 6, 6, 6, 4, 4],  # number of boxes per feature map location
}

In [ ]:
def multibox(vgg, extra_layers, cfg, num_classes):
    loc_layers = []
    conf_layers = []
    vgg_source = [21, -2]
    for k, v in enumerate(vgg_source):
        loc_layers += [nn.Conv2d(vgg[v].out_channels,
                                 cfg[k] * 4, kernel_size=3, padding=1)]
        conf_layers += [nn.Conv2d(vgg[v].out_channels,
                        cfg[k] * num_classes, kernel_size=3, padding=1)]
    for k, v in enumerate(extra_layers[1::2], 2):
        loc_layers += [nn.Conv2d(v.out_channels, cfg[k]
                                 * 4, kernel_size=3, padding=1)]
        conf_layers += [nn.Conv2d(v.out_channels, cfg[k]
                                  * num_classes, kernel_size=3, padding=1)]
    return vgg, extra_layers, loc_layers, conf_layers

#### **HYPER params**

In [ ]:
voc = {
    'num_classes': 21,
    'lr_steps': (80000, 100000, 120000),
    'max_iter': 200,
    'feature_maps': [38, 19, 10, 5, 3, 1],
    'min_dim': 300,
    'steps': [8, 16, 32, 64, 100, 300],
    'min_sizes': [30, 60, 111, 162, 213, 264],
    'max_sizes': [60, 111, 162, 213, 264, 315],
    'aspect_ratios': [[2], [2, 3], [2, 3], [2, 3], [2], [2]],
    'variance': [0.1, 0.2],
    'clip': True,
    'name': 'VOC',
}

#### **Building helpers**

In [ ]:
from __future__ import division
from math import sqrt as sqrt
from itertools import product as product

In [ ]:
class PriorBox(nn.Module):
    """Compute priorbox coordinates in center-offset form for each source
    feature map.
    """
    def __init__(self, cfg):
        super(PriorBox, self).__init__()
        self.image_size = cfg['min_dim']
        # number of priors for feature map location (either 4 or 6)
        self.num_priors = len(cfg['aspect_ratios'])
        self.variance = cfg['variance'] or [0.1]
        self.feature_maps = cfg['feature_maps']
        self.min_sizes = cfg['min_sizes']
        self.max_sizes = cfg['max_sizes']
        self.steps = cfg['steps']
        self.aspect_ratios = cfg['aspect_ratios']
        self.clip = cfg['clip']
        self.version = cfg['name']
        for v in self.variance:
            if v <= 0:
                raise ValueError('Variances must be greater than 0')

    def forward(self):
        with torch.no_grad():
            mean = []
            for k, f in enumerate(self.feature_maps):
                for i, j in product(range(f), repeat=2):
                    f_k = self.image_size / self.steps[k]
                    # unit center x,y
                    cx = (j + 0.5) / f_k
                    cy = (i + 0.5) / f_k

                    # aspect_ratio: 1
                    # rel size: min_size
                    s_k = self.min_sizes[k]/self.image_size
                    mean += [cx, cy, s_k, s_k]

                    # aspect_ratio: 1
                    # rel size: sqrt(s_k * s_(k+1))
                    s_k_prime = sqrt(s_k * (self.max_sizes[k]/self.image_size))
                    mean += [cx, cy, s_k_prime, s_k_prime]

                    # rest of aspect ratios
                    for ar in self.aspect_ratios[k]:
                        mean += [cx, cy, s_k*sqrt(ar), s_k/sqrt(ar)]
                        mean += [cx, cy, s_k/sqrt(ar), s_k*sqrt(ar)]
            # back to torch land
            output = torch.Tensor(mean).view(-1, 4)
            if self.clip:
                output.clamp_(max=1, min=0)
            return output

Custom L2Norm applied to conv4_3 output

In [ ]:
class L2Norm(nn.Module):
    def __init__(self,n_channels, scale):
        super(L2Norm,self).__init__()
        self.n_channels = n_channels
        self.gamma = scale or None
        self.eps = 1e-10
        self.weight = nn.Parameter(torch.Tensor(self.n_channels))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.constant_(self.weight,self.gamma)

    def forward(self, x):
        norm = x.pow(2).sum(dim=1, keepdim=True).sqrt()+self.eps
        #x /= norm
        x = torch.div(x,norm)
        out = self.weight.unsqueeze(0).unsqueeze(2).unsqueeze(3).expand_as(x) * x
        return out


decode - Decode locations from predictions using priors to undo
    the encoding we did for offset regression at train time.

In [ ]:
%cd ssd_pytorch

In [ ]:
from layers.box_utils import decode, nms

In [ ]:
class Detect(nn.Module):
    """At test time, Detect is the final layer of SSD.  Decode location preds,
    apply non-maximum suppression to location predictions based on conf
    scores and threshold to a top_k number of output predictions for both
    confidence score and locations.
    """
    def __init__(self, num_classes, bkg_label, top_k, conf_thresh, nms_thresh):
        super().__init__()
        self.num_classes = num_classes
        self.background_label = bkg_label
        self.top_k = top_k
        # Parameters used in nms.
        self.nms_thresh = nms_thresh
        if nms_thresh <= 0:
            raise ValueError('nms_threshold must be non negative.')
        self.conf_thresh = conf_thresh
        self.variance = voc['variance']

    def forward(self, loc_data, conf_data, prior_data):
        """
        Args:
            loc_data: (tensor) Loc preds from loc layers
                Shape: [batch,num_priors*4]
            conf_data: (tensor) Shape: Conf preds from conf layers
                Shape: [batch*num_priors,num_classes]
            prior_data: (tensor) Prior boxes and variances from priorbox layers
                Shape: [1,num_priors,4]
        """
        with torch.no_grad():
            num = loc_data.size(0)  # batch size
            num_priors = prior_data.size(0)
            output = torch.zeros(num, self.num_classes, self.top_k, 5)
            conf_preds = conf_data.view(num, num_priors,
                                        self.num_classes).transpose(2, 1)

            # Decode predictions into bboxes.
            for i in range(num):
                decoded_boxes = decode(loc_data[i], prior_data, self.variance)
                # For each class, perform nms
                conf_scores = conf_preds[i].clone()

                for cl in range(1, self.num_classes):
                    c_mask = conf_scores[cl].gt(self.conf_thresh)
                    scores = conf_scores[cl][c_mask]
                    if scores.size(0) == 0:
                        continue
                    l_mask = c_mask.unsqueeze(1).expand_as(decoded_boxes)
                    boxes = decoded_boxes[l_mask].view(-1, 4)
                    # idx of highest scoring and non-overlapping boxes per class
                    ids, count = nms(boxes, scores, self.nms_thresh, self.top_k)
                    output[i, cl, :count] = \
                        torch.cat((scores[ids[:count]].unsqueeze(1),
                                boxes[ids[:count]]), 1)
            flt = output.contiguous().view(num, -1, 5)
            _, idx = flt[:, :, 0].sort(1, descending=True)
            _, rank = idx.sort(1)
            flt[(rank < self.top_k).unsqueeze(-1).expand_as(flt)].fill_(0)
            return output

#### **Building**

In [ ]:
class SSD(nn.Module):
    """Single Shot Multibox Architecture
    The network is composed of a base VGG network followed by the
    added multibox conv layers.  Each multibox layer branches into
        1) conv2d for class conf scores
        2) conv2d for localization predictions
        3) associated priorbox layer to produce default bounding
           boxes specific to the layer's feature map size.
    See: https://arxiv.org/pdf/1512.02325.pdf for more details.

    Args:
        phase: (string) Can be "test" or "train"
        size: input image size
        base: VGG16 layers for input, size of either 300 or 500
        extras: extra layers that feed to multibox loc and conf layers
        head: "multibox head" consists of loc and conf conv layers
    """

    def __init__(self, phase, size, base, extras, loc_head, conf_head, num_classes, device):
        super(SSD, self).__init__()
        self.phase = phase
        self.num_classes = num_classes
        self.cfg = voc
        self.priorbox = PriorBox(self.cfg)
        # self.priors = Variable(self.priorbox.forward(), volatile=True)
        self.priors = self.priorbox.forward()
        self.priors = self.priors.to(device)
        self.size = size

        # SSD network
        self.vgg = nn.ModuleList(base)
        # Layer learns to scale the l2 normalized features from conv4_3
        self.L2Norm = L2Norm(512, 20)
        self.extras = nn.ModuleList(extras)

        self.loc = nn.ModuleList(loc_head)
        self.conf = nn.ModuleList(conf_head)

        if phase == 'test':
            self.softmax = nn.Softmax(dim=-1)
            self.detect = Detect(num_classes, 0, 200, 0.01, 0.45)

    def forward(self, x):
        """Applies network layers and ops on input image(s) x.

        Args:
            x: input image or batch of images. Shape: [batch,3,300,300].

        Return:
            Depending on phase:
            test:
                Variable(tensor) of output class label predictions,
                confidence score, and corresponding location predictions for
                each object detected. Shape: [batch,topk,7]

            train:
                list of concat outputs from:
                    1: confidence layers, Shape: [batch*num_priors,num_classes]
                    2: localization layers, Shape: [batch,num_priors*4]
                    3: priorbox layers, Shape: [2,num_priors*4]
        """
        sources = list()
        loc = list()
        conf = list()

        # apply vgg up to conv4_3 relu
        for k in range(23):
            x = self.vgg[k](x)

        s = self.L2Norm(x)
        sources.append(s)

        # apply vgg up to fc7
        for k in range(23, len(self.vgg)):
            x = self.vgg[k](x)
        sources.append(x)

        # apply extra layers and cache source layer outputs
        for k, v in enumerate(self.extras):
            x = F.relu(v(x), inplace=True)
            if k % 2 == 1:
                sources.append(x)

        # apply multibox head to source layers
        for (x, l, c) in zip(sources, self.loc, self.conf):
            loc.append(l(x).permute(0, 2, 3, 1).contiguous())
            conf.append(c(x).permute(0, 2, 3, 1).contiguous())

        loc = torch.cat([o.view(o.size(0), -1) for o in loc], 1)
        conf = torch.cat([o.view(o.size(0), -1) for o in conf], 1)
        if self.phase == "test":
            transformed_priors = self.priors.type(type(x.data)).to(device)
            output = self.detect(
                loc.view(loc.size(0), -1, 4),                   # loc preds
                self.softmax(conf.view(conf.size(0), -1,
                             self.num_classes)),                # conf preds
                transformed_priors                  # default boxes
            )
        else:
            output = (
                loc.view(loc.size(0), -1, 4),
                conf.view(conf.size(0), -1, self.num_classes),
                self.priors
            )
        return output

In [ ]:
def build_ssd(phase, size=300, num_classes=21):
    base_, extras_, loc_head_, conf_head_ = multibox(vgg(base[str(size)], 3),
                                     add_extras(extras[str(size)], 1024),
                                     mbox[str(size)], num_classes)
    return SSD(phase, size, base_, extras_, loc_head_, conf_head_, num_classes, device)

In [ ]:
ssd_net = build_ssd('test')

In [ ]:
ssd_net = ssd_net.to(device)

### **Look at it**

In [ ]:
ssd_net.state_dict()

### **Loading Process**

# **Model Convertation and Inference**

https://gitlab.deepschool.ru/a.kravchuk/guides/-/blob/main/_handbook/convertation/convertation.md

### **PyTorch Tools**

### **Tensors**

https://pytorch.org/docs/stable/notes/serialization.html

##### **Запомним Тензор**

In [ ]:
torch.save(torch.ones((2,2)), "tensor.pt")

In [ ]:
torch.load("tensor.pt")

Используем `weights_only=True` для повышения безопасности при "распаковке" (unpickling). Это ограничивает пул функций, исполнение которых возможно без дополнительных разрешений.
Параметр weights_only используется при загрузке тензоров с помощью torch.load() и позволяет ограничить десериализацию только загрузкой весов моделей, исключая выполнение произвольного кода. Это важная функция для безопасности, особенно при загрузке моделей из ненадёжных источников.

##### **Запомним python-словарь**

In [ ]:
d = {'layer1': torch.tensor([1, 2]).cuda(), 'layer2': torch.tensor([10, 20])}
torch.save(d, "tensor_dict.pt")

In [ ]:
# torch.load("tensor_dict.pt", weights_only=True)
torch.load("tensor_dict.pt", map_location='cuda', weights_only=True)

In [ ]:
torch.load("tensor_dict.pt", weights_only=True)

##### **Запомним тензоры, лежащие в одном месте в памяти (same storage)**

When PyTorch saves tensors it saves their storage objects and tensor metadata separately.

Only a single storage is written to 'small.pt’.

In [ ]:
large = torch.arange(0, 1000)
small = large[0:5]
torch.save(small, "small.pt")
loaded_small = torch.load("small.pt", weights_only=True)
loaded_small.untyped_storage().size()   # in bits

 Cloning a tensor produces a new tensor with a new storage object containing only the values in the tensor

In [ ]:
torch.save(small.clone(), "small.pt")
loaded_small = torch.load("small.pt", weights_only=True)
loaded_small.untyped_storage().size()   # in bits

Write answer

In [ ]:
numbers = torch.arange(1, 10)
evens = numbers[1::2]
torch.save([numbers, evens], 'tensors.pt')
loaded_numbers, loaded_evens = torch.load('tensors.pt', weights_only=True)
loaded_evens *= 2
loaded_numbers

### **Entire Model**

Сохранение модели целиком

In [ ]:
class MyModule(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.l0 = torch.nn.Linear(4, 2)
        self.l1 = torch.nn.Linear(2, 1)

    def forward(self, input):
        out0 = self.l0(input)
        out0_relu = torch.nn.functional.relu(out0)
        return self.l1(out0_relu)

In [ ]:
model = MyModule()
torch.save(model, "model.pth")

Загрузка модели целиком

In [ ]:
model_path = "model.pth"
torch.load(model_path, weights_only=False)

**Идея**: Под капотом PyTorch сериализует нашу модель с помощью `pickle` и сохраняет её на диск. `pickle` не сохраняет код класса модели, а лишь путь к файлу, в котором она описана.

**Недостаток**:
- Pickle не сохраняет код класса модели
- Pickle небезопасный


### **Weights Separately**  

Модель делится на две составляющие:
- **Класс модели** — описание слоёв и порядок их выполнения
- **Веса** — числа, которые перемножаются, складываются и пр.


Извлечение весов и постоянных буферов в виде python-словаря (module's state dict)

In [ ]:
bn = torch.nn.BatchNorm1d(3, track_running_stats=True)
print(list(bn.named_parameters()))
print(list(bn.named_buffers()))
bn.state_dict()

In [ ]:
state_dict = bn.state_dict()
torch.save(state_dict, 'bn.pth')

Сохранение module's state dict

Загрузка module's state dict

In [ ]:
state_dict = torch.load('bn.pth', weights_only=True)
new_bn = torch.nn.BatchNorm1d(3, track_running_stats=True)
new_bn.load_state_dict(state_dict)
new_bn.state_dict()

**Идея**: Разделение модели на структуру и наполнение.

**Недостаток**:
    Нам всё равно требуется код модели для её загрузки и инференса.

**Желание**

### **Let's use only PyTorch**

#### **Load SSD 300**

In [ ]:
model_path = "single_shot_detector.pth"
state_dict = torch.load(model_path, map_location='cpu', weights_only=True)
ssd_net.load_state_dict(state_dict)

In [ ]:
torch.serialization.add_safe_globals([SSD]) #, set, PriorBox])

Загрузка state_dict модели

In [ ]:
model_path = "single_shot_detector_state_dict.pth"
state_dict = torch.load(model_path, map_location='cpu', weights_only=True)
ssd_net.load_state_dict(state_dict)

#### **Inference with SSD 300**

##### **Add data**

In [ ]:
%cd ..

In [ ]:
image = cv2.imread("people.jpg")
image = cv2.resize(image, (300, 300))
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

In [ ]:
def normalize_input(image):
    imagenet_mean=(0.485, 0.456, 0.406)
    imagenet_std=(0.229, 0.224, 0.225)
    max_pixel_value=255.0

    transforms = A.Compose([
        A.Normalize(mean=imagenet_mean,
                    std=imagenet_std,
                    max_pixel_value=max_pixel_value,
                    normalization='standard',
                    p=1)
    ])

    transformed = transforms(image=image)
    return transformed["image"]

In [ ]:
transformed_image = normalize_input(image)
fig, ax = plt.subplots(1, figsize=(8, 8))
ax.imshow(transformed_image)

##### **Transform to PyTorch data**

In [ ]:
def numpy2torch(transformed_image):
    image_tensor = torch.Tensor(transformed_image)
    image_tensor = image_tensor.permute((2, 0, 1))
    image_tensor = image_tensor.unsqueeze(0)
    return image_tensor

In [ ]:
image_tensor = numpy2torch(transformed_image)
image_tensor = image_tensor.to(device)
image_tensor.shape

##### **Remember classes**

In [ ]:
VOC_CLASSES = (  # always index 0
    'aeroplane', 'bicycle', 'bird', 'boat',
    'bottle', 'bus', 'car', 'cat', 'chair',
    'cow', 'diningtable', 'dog', 'horse',
    'motorbike', 'person', 'pottedplant',
    'sheep', 'sofa', 'train', 'tvmonitor')

voc_classes_to_indexes = dict(zip(VOC_CLASSES, range(len(VOC_CLASSES))))
indexes_to_voc_classes = dict(zip(range(len(VOC_CLASSES)), VOC_CLASSES))

##### **Forward pass**

In [ ]:
with torch.no_grad():
    outputs = ssd_net(image_tensor)

In [ ]:
# skip j = 0, because it's the background class
print(f"outputs.shape: {outputs.shape}")

'''
torch.Size([1       - num of images in batch 
            21      - num of classes in net
            200     - num of bbs per class
            5       - [class_probability,
                       xmin / width,
                       ymin / height,
                       xmax / width,
                       ymax / height
                       ]
])
'''

##### **Post Processing**

In [ ]:
TP_boxes = []
image_size = (300, 300) # width, height
confidence_threshold = 0.4
num_classes = 21

In [ ]:
def post_processing(outputs):
    TP_boxes = []
    for j in range(1, num_classes):
        dets = outputs[0, j, :] # get detections of specified class label

        # take dets that probability of class greater than zero
        # from [num_bbs, 5]
        # dets[:, 0].gt(0.) -> [num_bbs]
        # .expand(5, dets.size(0)) -> [5, num_bbs]
        # .t() -> [num_bbs, 5]
        mask = dets[:, 0].gt(0.).expand(5, dets.size(0)).t()

        # get dets that probability of class > 0
        dets = torch.masked_select(dets, mask).view(-1, 5)
        
        if dets.size(0) == 0:
            # print(f"No detections for {indexes_to_voc_classes[num_classes]}")
            continue

        boxes = dets[:, 1:]
        scores = dets[:, 0].cpu().numpy()

        # to pixel coords
        for k in range(4):
            boxes[:, k] *= image_size[k % 2]
        
        # thresholding by confidence
        TP = scores > confidence_threshold

        # Detections with confidence > confidence_threshold exists
        if boxes[TP].numel():
            boxes_and_class = (boxes[TP].cpu().numpy(), j)
            TP_boxes.append(boxes_and_class)
    return TP_boxes

In [ ]:
def drawing(raw_image, TP_boxes):
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.5
    thickness = 2
    height, width, channels = raw_image.shape
    w_scale_f = width / image_size[0]
    h_scale_f = height / image_size[1]
    image = np.ascontiguousarray(raw_image)

    for boxes_array, class_id in TP_boxes:
        color_factor = int(255 * class_id / num_classes)
        color = (255 * color_factor,
                 255 * color_factor,
                 0
        )
        for box in boxes_array:
            xmin, ymin, xmax, ymax = box
            top_left = (int((xmin + 1) * w_scale_f),
                        int((ymin + 1) * h_scale_f)
            )
            bottom_right = (int((xmax - 1) * w_scale_f),
                            int((ymax - 1) * h_scale_f))

            cv2.rectangle(image, top_left, bottom_right, color, thickness)

            text = indexes_to_voc_classes[class_id - 1]
            text_coords = (top_left[0] + 5, top_left[1] + 10)
            cv2.putText(image, text, text_coords, font, font_scale, color, thickness)
    return image

In [ ]:
TP_boxes = post_processing(outputs)
resulting_img = drawing(image, TP_boxes)
plt.imshow(resulting_img)
plt.axis('off')
plt.show()

##### **Inference with webcam Processing**

In [ ]:
def pre_processing(frame):
    frame = cv2.resize(frame, image_size)
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    transformed_image = normalize_input(frame)
    image_tensor = numpy2torch(transformed_image)
    return image_tensor.to(device)

Code below should open GUI with image from webcam.

In [ ]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Could not open camera.")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to capture image.")
        break
    
    height, width, channels = frame.shape

    raw_image = deepcopy(frame)

    image_tensor = pre_processing(frame)
    with torch.no_grad():
        outputs = ssd_net(image_tensor)
    TP_boxes = post_processing(outputs)
    resulting_img = drawing(raw_image, TP_boxes)
    
    cv2.imshow("Camera", resulting_img)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


#### **SSD by NVIDIA**
https://pytorch.org/hub/nvidia_deeplearningexamples_ssd/

##### **Load model**

In [ ]:
import torch
ssd_model = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_ssd')
utils = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_ssd_processing_utils')
classes_to_labels = utils.get_coco_object_dictionary()

In [ ]:
ssd_model.to('cuda')
ssd_model.eval()

##### **Forward pass**

In [ ]:
with torch.no_grad():
    detections_batch = ssd_model(image_tensor)

##### **Post Processing**

In [ ]:
results_per_input = utils.decode_results(detections_batch)
best_results_per_input = [utils.pick_best(results, 0.40) for results in results_per_input]

In [ ]:
from matplotlib import pyplot as plt
import matplotlib.patches as patches

def nvidia_drawing(raw_image, best_results_per_input):
    image = np.ascontiguousarray(raw_image)
    height, width, channels = image.shape
    w_scale_f = width / image_size[0]
    h_scale_f = height / image_size[1]
    
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.5
    color = (255, 0, 0)
    thickness = 2

    
    for image_idx in range(len(best_results_per_input)):
        bboxes, classes, confidences = best_results_per_input[image_idx]
        for idx in range(len(bboxes)):
            left, bot, right, top = bboxes[idx]
            x, y, w, h = [val * 300 for val in [left, bot, right - left, top - bot]]
            top_left = (int(x * w_scale_f),
                        int(y * h_scale_f)
            )
            bottom_right = (int((x + w) * w_scale_f),
                            int((y + h) * h_scale_f)
            )
            cv2.rectangle(image, top_left, bottom_right, color, thickness)

            text_coords = (int(max(0, min(300, x * w_scale_f))),
                           int(max(0, min(300, (y - 5) * h_scale_f))))
            text = "{} {:.1f}%".format(classes_to_labels[classes[idx] - 1], confidences[idx]*100)
            cv2.putText(image, text, text_coords, font, font_scale, color, thickness)
    return image


In [ ]:
image_size = (300, 300)

In [ ]:
resulting_img = nvidia_drawing(image, best_results_per_input)
plt.imshow(resulting_img)
plt.axis('off')
plt.show()

##### **Inference with webcam Processing**

In [ ]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Could not open camera.")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to capture image.")
        break
    
    height, width, channels = frame.shape

    raw_image = deepcopy(frame)

    image_tensor = pre_processing(frame)
    
    with torch.no_grad():
        detections_batch = ssd_model(image_tensor)

    results_per_input = utils.decode_results(detections_batch)
    best_results_per_input = [utils.pick_best(results, 0.40) for results in results_per_input]    
    
    resulting_img = nvidia_drawing(raw_image, best_results_per_input)
    
    cv2.imshow("Camera", resulting_img)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


#### **Why not???**

**Недостатки**:
- `Нельзя установить PyTorch`  
Существуют вычислители, на которых нельзя установить наш фреймворк для инференса. А модельки выполнять хочется. Тогда, чаще всего, производитель железа пишет свой фреймворк для инференса и сетки выполняются на нём.
- `"Тяжеловесность" PyTorch`  
Фреймворки для обучения "тяжёлые". Во всех фреймворках для обучения есть много того, что нам не потребуется для инференса модели (расчёты градиентов, классы оптимизаторов и др.). Если мы будем выполнять разработку на встраиваемых устройствах, например FPGA, Jetson Nano, то установка PyTorch может съесть значительный кусок памяти.
- `Нет оптимизаций под определенное железо`  
Чаще всего, фреймворки для инференса пишут производители определённого железа. А кто кроме них может знать о том, как оптимальнее всего выполнять инференс на их железе?

Разделяют:
- `Фреймворки для обучения`  
PyTorch, TensorFlow, MXNet. На них мы только обучаем наши модели.
- `Фреймворки для инференса`  
ONNX Runtime, OpenVino, TensorRT, CoreML. Эти фреймворки используются только для инференса моделей.

### **TorchScript**

PyTorch format for inference with other programming languages

Разделяют:
- `Scripting (torch.jit.script)`  
компилятор, который выполняет прямой анализ исходного кода Python и преобразовывает его в промежуточное представление.
- `Tracing (torch.jit.trace)`  
инструмент для “захвата” вашей модели, т.е. “заморозка” графа выполнения.

#### **Scripting**

In [ ]:
scripted_model = torch.jit.script(ssd_model)

Преимущества:
- циклы и ветвления в графе выполнения продолжат работать как и прежде после конвертации модели

Недостатки:
- Не все операторы, используемые пользователем в forward подлежат трансляции из python в torchscript
- Примеры: numpy

Сохранение скриптованной модели

In [ ]:
torch.jit.save(scripted_model, 'model.pt')

Загрузка скриптованной модели

In [ ]:
new_scripted_model = torch.jit.load('model.pt', map_location='cpu')
new_scripted_model.to('cuda:0')

#### **Tracing**

In [ ]:
ssd_model = ssd_model.to('cuda:0')

In [ ]:
dummy_input = torch.rand(1, 3, 300, 300)
dummy_input = dummy_input.to('cuda:0')
traced_model = torch.jit.trace(ssd_model, dummy_input)

Преимущества:
- "Заморозка" графа выполнения позволяет обрабатывать даже те операции, что не подлежат трансляции в промежуточное представление. Происходит запоминание последовательности вычислений. Код описания модели не используется.

Недостатки:
- Получим модель, поток выполнения у которой будет заморожен. Т.е. как циклы и ветвления отработают на момент конвертации, так они и будут работать всегда.

Сохранение трэйсинг модели

In [ ]:
traced_model_path = 'model.pt'
torch.jit.save(traced_model, traced_model_path)

Загрузка трэйсинг модели

In [ ]:
new_traced_model = torch.jit.load(traced_model_path, map_location='cpu')
new_traced_model.to('cuda:0')

#### **Scripting and Tracing**

В модели есть ветвления или циклы, но есть и другие операции, которые не поддерживает скриптование? К нашему счастью, эти методы можно комбинировать. Например, можно выяснить, какая часть сети не поддерживает скриптование и затрейсить её. А потом уже для всей сети выполнить скриптование.



#### **Inference with TorchScript**

In [ ]:
import torch

scripted_model = torch.jit.load(model_path, map_location='cpu')
scripted_model.to('cuda:0')

with torch.no_grad():
    result = scripted_model(torch.rand(1, 3, 224, 224))

In [ ]:
import torch

traced_model = torch.jit.load(traced_model_path, map_location='cpu')
traced_model.to('cuda:0')

with torch.no_grad():
    result = traced_model(torch.rand(1, 3, 300, 300, device=device))


In [ ]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Could not open camera.")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to capture image.")
        break
    
    height, width, channels = frame.shape

    raw_image = deepcopy(frame)

    image_tensor = pre_processing(frame)
    
    with torch.no_grad():
        detections_batch = traced_model(image_tensor)

    results_per_input = utils.decode_results(detections_batch)
    best_results_per_input = [utils.pick_best(results, 0.40) for results in results_per_input]    
    
    resulting_img = nvidia_drawing(raw_image, best_results_per_input)
    
    cv2.imshow("Camera", resulting_img)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


TorchScript - это промежуточное представление PyTorch моделей, градиенты по-прежнему считаются.
Поэтому нужно не забывать оборачивать инференс в контекстный менеджер `torch.no_grad()`.


### **ONNX**

ONNX - открытый формат для представления моделей машинного обучения (не только нейронных сетей, но и классических моделей).

ONNX - мост между различными фреймворками.

**Model Portability**:
 - Конвертация моделей:  
   - Из любого фреймворка в ONNX
   - Из ONNX в любой фреймворк
 - Кросс-платформенность:  
   - Серверные решения (CPU, GPU)
   - Мобильные устройства (iOS, Android)
   - Встроенные системы (edge devices, IoT)

**Efficiency**:
 - Оптимизация моделей
   - Графовых оптимизации
     - Constant Folding
     - Redundant node eliminations (Dropout)
     - Semantics-preserving node fusions (Conv Add Fusion, Conv Mul Fusion)
   - Квантования
 - Поддержка аппаратного ускорения:
   - **CPU**: С оптимизациями для Intel, AMD, Arm
   - **GPU**: С поддержкой CUDA, TensorRT, DirectML.
   - **Специализированных ускорителей**: Например, Intel OpenVINO, NVIDIA TensorRT  
   
This efficiency is crucial for real-world applications where computational resources may be limited.

ONNX computational graph is adaptable with lots of DL frameworks. Consist of:
 - nodes (operations)
   - Математические операции: Add, Sub, Mul, Div.
   - Линейная алгебра: MatMul, Gemm.
   - Свертки: Conv, Pooling.
   - Активации: Relu, Sigmoid, Tanh.
   - Манипуляции с данными: Reshape, Transpose, Concat, Split.
 - edges (tensors)

**ONNX operators**
https://onnx.ai/onnx/operators/index.html  

Custom operators can be defined

Nodes include:
 - **initializers** (constant tensors)
   - Веса сети.
   - Параметры нормализации (BatchNorm)
 - **attributes** (hyperparams of operations)
   - Для **Conv** (kernel_shape, strides, pads)

##### **Building ONNX model**

In [ ]:
import onnx

In [ ]:
matmul_node = onnx.helper.make_node(
    "MatMul",        # Operation type
    ["input1", "input2"],  # Input tensors
    ["matmul_output"],      # Output tensor
)

relu_node = onnx.helper.make_node(
    "Relu",          # Operation type
    ["matmul_output"],  # Input
    ["output"],      # Output
)

In [ ]:
input1 = onnx.helper.make_tensor_value_info("input1", onnx.TensorProto.FLOAT, [3, 3])
input2 = onnx.helper.make_tensor_value_info("input2", onnx.TensorProto.FLOAT, [3, 3])
output = onnx.helper.make_tensor_value_info("output", onnx.TensorProto.FLOAT, [3, 3])

In [ ]:
graph = onnx.helper.make_graph(
    [matmul_node, relu_node],  # Nodes
    "simple_graph",            # Graph name
    [input1, input2],          # Inputs
    [output],                  # Outputs
)

In [ ]:
model = onnx.helper.make_model(graph)
onnx.save(model, "simple_model.onnx")

**Model Visualization**

In [ ]:
!pip install netron

In [ ]:
import netron
netron.start("simple_model.onnx")

#### **PyTorch -> ONNX**

In [ ]:
torch.onnx.export(
    model,                                      # PyTorch-модель
    dummy_input,                 # входной тензор нужного размера
    'model.onnx',                               # путь куда сохранить модель в ONNX-формате.
    verbose=True,                               # опционально
    input_names=['input'],                      # опционально
    output_names=['output'],                    # опционально
    dynamic_axes={'input': [0], 'output': [0]}, # опционально 
)

- `verbose`: хотим ли видеть "отчёт" экспорта;

- `input_names` и `output_names`: имена входных и выходных нод, может быть полезно для дальнейшего использования модели, см. ниже;

- `dynamic_axes`: указываем динамические оси, в нашем случае 0 - размер батча;
по умолчанию onnx граф статический и если нам нужны динамические оси - необходимо указать это явно (полезно для конвертации в TensorRT с динамическим размером батча).

- `opset_version`: версия множества операторов onnx для экспорта модели. Обычно этот параметр редко отличается от дефолтного и если его нужно изменить, в логах вам об этом напомнят.
Как правило, PyTorch опаздывает от самой актуальной версии opset ONNX. Например, по умолчанию стоит 9 версия, а уже можно пользовать 11.
Скорее всего, вам редко понадобится изменять этот параметр, но знать о нём полезно. В случае проблем с экспортом "новых" слоёв, можно использовать новые нестабильные версии.

**Dynamic Shapes (Динамические формы)**

ONNX поддерживает динамические формы тензоров, что позволяет моделям работать с данными переменного размера. Это особенно полезно в реальных сценариях, где размер входных данных может меняться. Например:

- **Динамический размер батча**: Модель может обрабатывать данные с разным размером батча. Это полезно, когда размер батча может варьироваться в зависимости от доступных ресурсов или требований задачи.

- **Динамические размеры изображений**: Модель может работать с изображениями разного разрешения. Это важно, например, в задачах компьютерного зрения, где изображения могут иметь различные размеры (биомед).

#### **PyTorch to ONNX export of ResNet50**

In [ ]:
from torchvision import models, datasets, transforms as T
from PIL import Image
import onnx

In [ ]:
resnet50 = models.resnet50(pretrained=True)

In [ ]:
resnet50 = resnet50.to(device)

In [ ]:
image_height = 224
image_width = 224
x = torch.randn(1, 3, image_height, image_width, requires_grad=True)
x = x.to(device)

torch.onnx.export(resnet50,                     # model being run
                  x,                            # model input (or a tuple for multiple inputs)
                  "resnet50.onnx",              # where to save the model (can be a file or file-like object)
                  export_params=True,           # store the trained parameter weights inside the model file
                  opset_version=12,             # the ONNX version to export the model to
                  do_constant_folding=True,     # whether to execute constant folding for optimization
                  input_names = ['input'],      # the model's input names
                  output_names = ['output'],    # the model's output names
)


In [ ]:
onnx.checker.check_model("resnet50.onnx")

Compare PyTorch and ONNX computational graphs - https://netron.app/

#### **ONNX Runtime**

**Фреймворк для инференса**

Преимущества:
- `Ускорение инференса`  
ONNX Runtime делает инференс значительно быстрее, чем исходный PyTorch.
- `Инференс на различных языках программирования, различных устройствах и бэкендах`  
Сейчас на официальной странице указано 8 языков программирования, 5 архитектур вычислителей и 19 бэкендов.
Более того, разработчики ONNX Runtime и энтузиасты постоянно расширяют возможности библиотеки и некоторые вещи ещё не имеют официальной поддержки, но уже работают.

#### **ONNX Runtime Installation**

Ваш ноутбук на архитектуре процессора x64

In [ ]:
%pip install onnxruntime-gpu

#### **ONNX Runtime Inference**

Execution Providers - интерфейсы для аппаратного ускорения. Позволяют запрашивать аппаратные возможности.

Execution Providers:  
 - **Kernel-based** (CPUExecutionProvider, CUDAExecutionProvider):
   - Прямое взаимодействие с аппаратным обеспечением (CUDA, cuDNN, TensorRt, OpenVINO)
   - Требует настройки и установки дополнительных зависимостей
 - **Runtime-based** (nGraphExecutionProvider, TensorRTExecutionProvider)
   - Взаимодействие с железом через внешние runtinme-библиотеки (дополнительный слой абстракции)
   - Кросс-платформенность, простота использования
   - Поддерживают меньше низкоуровневых оптимизаций


##### **Compare PyTorch and ONNX forward pass**

**Preprocessing**

In [ ]:
filename = 'people.jpg' # change to your filename
input_image = Image.open(filename)

preprocess = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
input_tensor = preprocess(input_image)
input_batch = input_tensor.unsqueeze(0)

**PyTorch CPU**

In [ ]:
resnet50.eval()
input_batch = input_batch.to('cpu')
resnet50.to('cpu')
input_batch.shape

In [ ]:
%%timeit
with torch.no_grad():
    output = resnet50(input_batch)

**PyTorch GPU**

In [ ]:
input_batch = input_batch.to(device)
resnet50.to(device)
input_batch.shape

In [ ]:
%%timeit
with torch.no_grad():
    output = resnet50(input_batch)

https://docs.nvidia.com/deeplearning/cudnn/installation/latest/linux.html#ubuntu-debian-local-installation

!apt-cache search cudnn

https://onnxruntime.ai/docs/install/

In [ ]:
import onnxruntime as ort

In [ ]:
ort.get_available_providers()

**ONNXRuntime CPU**

ONNX Runtime expects numpy arrays

In [ ]:
session_fp32 = ort.InferenceSession("resnet50.onnx", providers=['CPUExecutionProvider'])
input_arr = input_batch.cpu().detach().numpy()
input_arr.shape

In [ ]:
%%timeit
ort_outputs = session_fp32.run([], {'input':input_arr})[0]

**ONNXRuntime GPU**

In [ ]:
session_fp32 = ort.InferenceSession("resnet50.onnx", providers=['CUDAExecutionProvider'])
ort_value = ort.OrtValue.ortvalue_from_numpy(input_arr, 'cuda', 0)

In [ ]:
%%timeit
ort_outputs = session_fp32.run([], {'input':ort_value})[0]

**ONNXRuntime GPU Graph Optimizations**

In [ ]:
session_options = ort.SessionOptions()
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
session = ort.InferenceSession("resnet50.onnx", session_options, providers=['CUDAExecutionProvider'])

In [ ]:
%%timeit
ort_outputs = session.run([], {'input':ort_value})[0]

**ONNXRuntime GPU IO Bindings**

In [ ]:
print([input_.name for input_ in session.get_inputs()])
print([output_.name for output_ in session.get_outputs()])

In [ ]:
input_name = 'input'
output_name = 'output'
ort_outputs = session.run([], {'input':ort_value})[0]

In [ ]:
io_binding = session.io_binding()
io_binding.bind_input(input_name, 'cuda', 0, np.float32, input_arr.shape, ort_value.data_ptr())
io_binding.bind_output(output_name, 'cuda', 0, np.float32, ort_outputs.shape)

In [ ]:
%%timeit
session.run_with_iobinding(io_binding)

**ONNXRuntime ORT format**

The ORT format is the format supported by reduced size ONNX Runtime builds. Reduced size builds may be more appropriate for use in size-constrained environments such as mobile and web applications

In [ ]:
!python3 -m onnxruntime.tools.convert_onnx_models_to_ort "../resnet50.onnx"

No python API

### **OpenVINO**

**Efficiency**:
 - Оптимизация моделей
   - Графовых оптимизации
     - Constant Folding
     - Redundant node eliminations (Dropout)
     - Semantics-preserving node fusions (Conv Add Fusion, Conv Mul Fusion)
   - Квантования
 - Поддержка аппаратного ускорения Intel:
   - **Intel CPU**
   - **iGPU**
   - **VPU**: Например, Intel Movidius  
   - **FPGA**: Например, Intel Arria

https://docs.openvino.ai/2025/get-started/learn-openvino/interactive-tutorials-python.html

#### **Installation with ubuntu 22**

wget https://apt.repos.intel.com/intel-gpg-keys/GPG-PUB-KEY-INTEL-SW-PRODUCTS.PUB &&
sudo gpg --output /etc/apt/trusted.gpg.d/intel.gpg --dearmor GPG-PUB-KEY-INTEL-SW-PRODUCTS.PUB

echo "deb https://apt.repos.intel.com/openvino/2025 ubuntu22 main" | sudo tee /etc/apt/sources.list.d/intel-openvino-2025.list

sudo apt update &&
sudo apt install openvino

pip install openvino

#### **Convert a Model to IR representation**

In [ ]:
import openvino as ov
import logging as log
import sys

In [ ]:
input_data = torch.rand(1, 3, 224, 224)
ov_model = ov.convert_model(resnet50, example_input=input_data)

In [ ]:
ov.save_model(ov_model, 'model.xml')

**Initialize OpenVINO Runtime Core**

In [ ]:
core = ov.Core()

**Initialize OpenVINO Runtime Core**

In [ ]:
core.available_devices

In [ ]:
# https://docs.openvino.ai/2025/get-started/learn-openvino/openvino-samples/hello-query-device.html
def param_to_string(parameters) -> str:
    """Convert a list / tuple of parameters returned from OV to a string."""
    if isinstance(parameters, (list, tuple)):
        return ', '.join([str(x) for x in parameters])
    else:
        return str(parameters)


def hello_query_device():
    log.basicConfig(format='[ %(levelname)s ] %(message)s', level=log.INFO, stream=sys.stdout)

    # --------------------------- Step 1. Initialize OpenVINO Runtime Core --------------------------------------------
    core = ov.Core()

    # --------------------------- Step 2. Get metrics of available devices --------------------------------------------
    log.info('Available devices:')
    for device in core.available_devices:
        log.info(f'{device} :')
        log.info('\tSUPPORTED_PROPERTIES:')
        for property_key in core.get_property(device, 'SUPPORTED_PROPERTIES'):
            if property_key not in ('SUPPORTED_PROPERTIES'):
                try:
                    property_val = core.get_property(device, property_key)
                except TypeError:
                    property_val = 'UNSUPPORTED TYPE'
                log.info(f'\t\t{property_key}: {param_to_string(property_val)}')
        log.info('')

    # -----------------------------------------------------------------------------------------------------------------
    return 0

In [ ]:
hello_query_device()
device = "CPU"

**prepare to inference**

In [ ]:
compiled_model = ov.compile_model(ov_model, device)

**OpenVINO CPU**

In [ ]:
%%timeit
result = compiled_model(input_data)

In [ ]:
import openvino.properties as props
import openvino.properties.hint as hints

In [ ]:
config = {hints.performance_mode: hints.PerformanceMode.LATENCY}
compiled_model = core.compile_model(ov_model, "CPU", config)

In [ ]:
%%timeit
result = compiled_model(input_data)

In [ ]:
core = ov.Core()
core.set_property(
    "CPU",
    {hints.execution_mode: hints.ExecutionMode.PERFORMANCE},
)
config = {hints.performance_mode: hints.PerformanceMode.LATENCY}
compiled_model = core.compile_model(ov_model, "CPU", config)

In [ ]:
%%timeit
result = compiled_model(input_data)

In [ ]:
infer_request = compiled_model.create_infer_request()
tensor2 = ov.Tensor(ov.Type.f32, [1, 3, 224, 224])
infer_request.set_tensor("x", tensor2)

In [ ]:
%%timeit
infer_request.infer()

In [ ]:
!netron model.xml